# Suite TS — The TypeScript language

TypeScript and JavaScript as one tree language (`@mbse/programs/TypeScript/Syntax`), whose kinds follow
typescript-estree: its kinds and categories, the identifiers it accepts, each kind's own checks, and where each kind
and feature exists, in ECMAScript's editions and TypeScript's versions, which the standards check.

In [ ]:
import { Syntax as F } from "@mbse/programs/Framework";
import { ECMAScript2020, ECMAScript2025, Syntax as S, TypeScript50, TypeScript59 } from "@mbse/programs/TypeScript";
import { ECMAScriptStandard, TypeScriptStandard } from "@mbse/programs/TypeScript/_Standard";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const n = (name: unknown) => new S.Identifier({ name } as object);
const lit = (raw: unknown) => new S.Literal({ raw } as object);
const annotated = (name: string, type: F.Node) =>
  new S.Identifier({ name, typeAnnotation: new S.TSTypeAnnotation({ typeAnnotation: type }) });

## TS-01 · Every kind belongs to the language, the grammar lists them, and their fields are in source order

In [ ]:
const kinds = L.kinds();
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
assert(kinds.size === 163);
equal([...L.categories().keys()], ["Statement", "Expression", "Pattern", "ClassElement", "TypeElement", "TypeNode"]);
assert(S.KINDS.every((k) => k.NAME === `Programs.TypeScript.${k.KIND}`));
const grammar = L.grammar() as any;
assert(grammar.language === "TypeScript");
equal(grammar.base, { ECMAScript: 2015, TypeScript: 100 });
const satisfies = grammar.kinds.find((k: any) => k.kind === "TSSatisfiesExpression");
equal(satisfies.since, { TypeScript: 409 });
assert(satisfies.category === "Expression");
assert(grammar.kinds.find((k: any) => k.kind === "JSXElement").extension === true);
equal(S.KINDS.filter((k) => k.EXTENSION).map((k) => k.name), S.KINDS.filter((k) => k.name.startsWith("JSX")).map((k) => k.name));
equal(S.es(2020, 307), { ECMAScript: 2020, TypeScript: 307 });
equal(S.ts(509), { TypeScript: 509 });
assert(new S.Comment() instanceof S.Statement && new S.TSAnyKeyword() instanceof S.TypeNode);
equal(S.ClassDeclaration.FIELDS.map((f) => f.name), [
  "decorators", "declare", "abstract", "id", "typeParameters", "superClass", "superTypeArguments", "implements", "body"]);
equal(S.MethodDefinition.FIELDS.map((f) => f.name), [
  "decorators", "accessibility", "static", "override", "methodKind", "computed", "key", "optional", "value"]);
equal(S.Identifier.FIELDS.map((f) => f.name), ["decorators", "name", "optional", "typeAnnotation"]);
console.log(kinds.size, "kinds");

## TS-02 · Identifiers are Unicode identifiers, `$` included; keywords are names to the tree

In [ ]:
for (const name of ["x_1", "$", "$el", "_", "Größe", "async", "of", "class", "this"]) {
  equal(L.validate(n(name)), []);
  equal(L.validate(new S.PrivateIdentifier({ name })), []);
}
equal(L.validate(n("1x")), ["'1x' is not an identifier"]);
equal(L.validate(n("a-b")), ["'a-b' is not an identifier"]);
equal(L.validate(n("")), ["'' is not an identifier"]);
equal(L.validate(new S.PrivateIdentifier({ name: "#x" })), ["'#x' is not an identifier"]);
equal(L.validate(n(3n)), ["Identifier.name must be a str, got int"]);
equal(L.validate(new S.PrivateIdentifier({ name: 3n } as object)), ["PrivateIdentifier.name must be a str, got int"]);
equal(L.validate(new S.Identifier()), ["an Identifier needs a name"]);
console.log("ok");

## TS-03 · Each kind's own checks: what its lists must hold, and what goes together

In [ ]:
const loop = new S.WhileStatement({ test: n("a"), body: new S.EmptyStatement() });
const problems: [F.Node, string][] = [
  [lit(""), "a Literal needs a raw"],
  [new S.TemplateLiteral(), "a TemplateLiteral needs one more quasi than expressions"],
  [new S.TemplateLiteral({ quasis: [new S.TemplateElement({ raw: "a" })], expressions: [n("b")] }),
    "a TemplateLiteral needs one more quasi than expressions"],
  [new S.SequenceExpression({ expressions: [n("a")] }), "a SequenceExpression needs two expressions"],
  [new S.TryStatement({ block: new S.BlockStatement() }), "a TryStatement needs a handler or a finalizer"],
  [new S.VariableDeclaration({ declarationKind: "let" }), "a VariableDeclaration needs a declarator"],
  [new S.VariableDeclarator({ id: n("a"), definite: true }), "definite needs a name with a type annotation"],
  [new S.TSTypeParameterDeclaration(), "a TSTypeParameterDeclaration needs a type parameter"],
  [new S.TSTemplateLiteralType(), "a TSTemplateLiteralType needs one more quasi than types"],
  [new S.TSUnionType({ types: [new S.TSAnyKeyword()] }), "a TSUnionType needs two types"],
  [new S.TSIntersectionType({ types: [new S.TSAnyKeyword()] }), "a TSIntersectionType needs two types"],
];
for (const [node, problem] of problems) equal(L.validate(node), [problem]);
// `if (a) if (b) x; else y;` would give the else to the inner `if`, through loops, labels and `with` too
const dangling = "an if without else ends the consequent, and would take the else";
for (const consequent of [
  new S.IfStatement({ test: n("b"), consequent: new S.EmptyStatement() }),
  new S.LabeledStatement({ label: n("l"), body: new S.ForOfStatement({
    left: n("x"), right: n("y"), body: new S.IfStatement({ test: n("b"), consequent: new S.EmptyStatement() }) }) }),
  new S.IfStatement({ test: n("b"), consequent: new S.EmptyStatement(), alternate: new S.WithStatement({
    object: n("o"), body: new S.IfStatement({ test: n("c"), consequent: new S.EmptyStatement() }) }) }),
]) {
  equal(L.validate(new S.IfStatement({ test: n("a"), consequent, alternate: new S.EmptyStatement() })), [dangling]);
}
const fine = [
  new S.IfStatement({ test: n("a"), consequent: loop, alternate: new S.EmptyStatement() }),
  new S.IfStatement({ test: n("a"), consequent: new S.IfStatement({ test: n("b"), consequent: new S.EmptyStatement(),
    alternate: new S.EmptyStatement() }), alternate: new S.EmptyStatement() }),
  new S.IfStatement({ test: n("a"), consequent: new S.IfStatement({ test: n("b"), consequent: new S.EmptyStatement() }) }),
  new S.TemplateLiteral({ quasis: [new S.TemplateElement({ raw: "a" }), new S.TemplateElement({ raw: "b", tail: true })],
    expressions: [n("b")] }),
  new S.TryStatement({ block: new S.BlockStatement(), finalizer: new S.BlockStatement() }),
  new S.VariableDeclarator({ id: annotated("a", new S.TSNumberKeyword()), definite: true }),
  new S.TSUnionType({ types: [new S.TSAnyKeyword(), new S.TSNullKeyword()] }),
];
for (const node of fine) equal(L.validate(node), []);
equal(L.validate(lit(1n)), ["Literal.raw must be a str, got int"]);
equal(L.validate(new S.Literal()), ["a Literal needs a raw"]);
equal(lit(1n).features(), []);
console.log(problems.length, "problems");

## TS-04 · Where kinds and features exist: since which edition of ECMAScript, and which version of TypeScript

In [ ]:
type Standard = { check(node: F.Node): string[]; name(): string };
/** `node` is fine in the standard `fine`, and has `problem` in the standard `lacking`. */
function needs(node: F.Node, problem: string, fine: Standard, lacking: Standard): void {
  equal(fine.check(node), []);
  equal(lacking.check(node), [problem]);
}

assert(TypeScript59.STANDARD.name() === "TypeScript 5.9" && TypeScript59.JSX.name() === "TypeScript 5.9 with JSX");
assert(ECMAScript2025.STANDARD.name() === "ES2025" && ECMAScript2020.JSX.name() === "ES2020 with JSX");
assert(TypeScript50.STANDARD.toString() === "<standard TypeScript 5.0>");
assert(TypeScript59.STANDARD.version === 509 && ECMAScript2025.STANDARD.version === 2025);
const es = (year: number) => new ECMAScriptStandard(year);
const ts = (major: number, minor: number) => new TypeScriptStandard(major, minor);
const [ES2015, ES2016, ES2019, ES2021, ES2022] = [es(2015), es(2016), es(2019), es(2021), es(2022)];
const [ES2020, ES2025] = [ECMAScript2020.STANDARD, ECMAScript2025.STANDARD];
const [TS44, TS49, TS50] = [ts(4, 4), ts(4, 9), ts(5, 0)];
const TS59 = TypeScript59.STANDARD;
const any = () => new S.TSAnyKeyword();
needs(any(), "TSAnyKeyword is not ECMAScript", TS59, ES2025);
needs(new S.TSSatisfiesExpression({ expression: n("a"), typeAnnotation: any() }),
  "TSSatisfiesExpression needs TypeScript 4.9", TS49, ts(4, 8));
needs(new S.PrivateIdentifier({ name: "x" }), "PrivateIdentifier needs ES2022", ES2022,
  ES2021);
const es2019 = es(2019);
needs(new S.ChainExpression({ expression: n("a") }), "ChainExpression needs ES2020", ES2020, es2019);
needs(new S.BinaryExpression({ left: n("a"), operator: "**", right: n("b") }),
  "BinaryExpression.operator '**' needs ES2016", ES2016, ES2015);
needs(new S.AssignmentExpression({ left: n("a"), operator: "??=", right: n("b") }),
  "AssignmentExpression.operator '??=' needs ES2021", ES2021, ES2020);
needs(new S.LogicalExpression({ left: n("a"), operator: "??", right: n("b") }),
  "LogicalExpression.operator '??' needs ES2020", ES2020, es2019);
needs(lit("1n"), "bigint Literal needs ES2020", ES2020, es2019);
needs(lit("1_000"), "Literal with numeric separators needs ES2021", ES2021,
  ES2020);
assert(ES2015.check(lit("1.5")).length === 0 && ES2015.check(lit("'a_n'")).length === 0);
equal(ES2015.check(lit("._5")), ["Literal with numeric separators needs ES2021"]);
needs(new S.MetaProperty({ meta: n("import"), property: n("meta") }), "import.meta needs ES2020",
  ES2020, es2019);
equal(ES2015.check(new S.MetaProperty({ meta: n("new"), property: n("target") })), []);
equal(ES2015.check(new S.MetaProperty()), []);
needs(new S.MemberExpression({ object: n("a"), property: n("b"), optional: true }),
  "MemberExpression with ?. needs ES2020", ES2020, es2019);
needs(new S.CallExpression({ callee: n("a"), optional: true }), "CallExpression with ?. needs ES2020",
  ES2020, es2019);
const privateIn = new S.BinaryExpression({ left: new S.PrivateIdentifier({ name: "x" }), operator: "in", right: n("o") });
equal((ES2022).check(privateIn), []);
equal((ES2021).check(privateIn), [
  "BinaryExpression of #name in needs ES2022", "left: PrivateIdentifier needs ES2022"]);
needs(new S.CatchClause({ body: new S.BlockStatement() }), "CatchClause without a param needs ES2019",
  ES2019, es(2018));
needs(new S.VariableDeclaration({ declarationKind: "using", declarations: [new S.VariableDeclarator({ id: n("a") })] }),
  "VariableDeclaration.declarationKind 'using' is not ECMAScript", TS59, ES2025);
needs(new S.ImportDeclaration({ source: lit("'m'"), phase: "defer" }),
  "ImportDeclaration.phase 'defer' needs TypeScript 5.9", TS59, TS50);
needs(new S.Identifier({ name: "a", optional: true }), "Identifier.optional is not ECMAScript", TS59,
  ES2025);
needs(new S.ArrayPattern({ optional: true }), "ArrayPattern.optional is not ECMAScript", TS59, ES2025);
needs(new S.TSTypeParameter({ name: n("T"), isIn: true }), "TSTypeParameter.isIn needs TypeScript 4.7",
  TS49, TS44);
// JSX is an extension, outside every edition and version: a standard has it with `jsx`
equal(ES2015.check(new S.JSXText({ raw: "a" })), ["JSXText needs JSX"]);
equal(ES2015.problems(new S.JSXText({ raw: "a" })), ["JSXText needs JSX"]);
equal(new ECMAScriptStandard(2015, true).problems(new S.JSXText({ raw: "a" })), []);
const cast = new S.TSTypeAssertion({ typeAnnotation: any(), expression: n("a") });
equal(TypeScript59.JSX.problems(cast), ["TSTypeAssertion is not allowed with JSX"]);
equal(TS59.problems(cast), []);
equal(ts(4, 8).check(new S.Program({ body: [new S.ExpressionStatement({
  expression: new S.TSSatisfiesExpression({ expression: n("a"), typeAnnotation: any() }) })] })), [
  "body[0].expression: TSSatisfiesExpression needs TypeScript 4.9"]);
console.log("ok");